# Chapter 6 — Advanced Logistic Regression and Extensions

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Machine Learning  
**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas logistic regression lanjutan untuk klasifikasi biner, multiclass, dan multilabel. Materi meliputi overview logistic regression, regularization, strategi multiclass, multilabel classification, serta evaluasi model.

## Theory — Logistic Regression

Logistic Regression digunakan untuk memodelkan probabilitas kelas. Pada klasifikasi biner, kombinasi linear fitur diubah menjadi probabilitas menggunakan fungsi sigmoid. Model memperkirakan koefisien berdasarkan data training.

Berbeda dari Linear Regression yang memprediksi nilai kontinu, Logistic Regression digunakan untuk target kategorikal. Evaluasi sebaiknya tidak hanya melihat accuracy, tetapi juga precision, recall, F1, confusion matrix, dan ROC-AUC sesuai kebutuhan.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, RocCurveDisplay
)

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Logistic Regression — ROC Curve")
plt.show()

## Theory — Multiclass Classification

Multiclass classification memprediksi satu kelas dari lebih dari dua kelas.
Strategi yang umum adalah multinomial logistic regression dan One-vs-Rest (OvR).
Pilihan strategi dapat memengaruhi cara model membentuk batas keputusan.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.multiclass import OneVsRestClassifier

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Multinomial": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            solver="lbfgs", max_iter=3000, random_state=42
        ))
    ]),
    "One-vs-Rest": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", OneVsRestClassifier(
            LogisticRegression(max_iter=3000, random_state=42)
        ))
    ])
}

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    print(name, "accuracy:", round(
        accuracy_score(y_test, estimator.predict(X_test)), 4
    ))

## Theory — Regularization

Regularization membantu mengendalikan besarnya koefisien agar model tidak
terlalu menyesuaikan diri dengan data training. Dalam Logistic Regression
scikit-learn, parameter `C` adalah kebalikan kekuatan regularization: semakin
kecil `C`, semakin kuat regularization.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42))
])
param_grid = {"classifier__C": [0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Test ROC-AUC:", round(
    roc_auc_score(y_test, search.predict_proba(X_test)[:, 1]), 4
))

## Theory — Multilabel Classification

Dalam multilabel classification, satu observasi dapat memiliki lebih dari satu
label secara bersamaan. Ini berbeda dari multiclass classification, yang
memilih satu kelas dari sejumlah kelas yang saling eksklusif.

In [ ]:
from sklearn.datasets import make_multilabel_classification
from sklearn.metrics import f1_score, hamming_loss

X_ml, y_ml = make_multilabel_classification(
    n_samples=1000, n_features=20, n_classes=4,
    n_labels=2, random_state=42
)
Xtr, Xte, ytr, yte = train_test_split(
    X_ml, y_ml, test_size=0.2, random_state=42
)

multilabel_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", OneVsRestClassifier(
        LogisticRegression(max_iter=3000, random_state=42)
    ))
])
multilabel_model.fit(Xtr, ytr)
pred = multilabel_model.predict(Xte)

print("Exact-match accuracy:", round(accuracy_score(yte, pred), 4))
print("Micro F1:", round(f1_score(yte, pred, average="micro"), 4))
print("Hamming loss:", round(hamming_loss(yte, pred), 4))

## Analysis and Conclusion

- Accuracy mengukur proporsi prediksi benar.
- Precision dan recall membantu menganalisis jenis kesalahan klasifikasi.
- ROC-AUC mengukur kemampuan model membedakan kelas berdasarkan skor.
- Multiclass dan multilabel merupakan dua skenario target yang berbeda.
- Regularization mengendalikan kompleksitas; nilai `C` yang lebih kecil berarti regularization lebih kuat.

**Conclusion:** Logistic Regression dapat digunakan untuk klasifikasi biner,
multiclass, dan multilabel. Pemilihan metrik evaluasi perlu disesuaikan dengan
karakteristik data dan tujuan klasifikasi.